# TF Expression vs TF Activity — figure-only notebook

This notebook **does not rerun glmnet or LOOCV**. It reads the plot-ready TSV files written by the main analysis script and recreates the final figures.

- Model tuning remains pooled OOF **AUPR**.
- Final figures include both **PR/AUPR** and **ROC/AUC**.
- Change the figure-size variables below to control both notebook display size and saved PDF/PNG size.
- The last section plots one selected celltype-region ROC/AUC figure.


In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

# ============================================================
# USER SETTINGS
# ============================================================

output_dir <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/07_tf/figures"

# Figures regenerated by this notebook will be saved here.
figure_dir <- file.path(output_dir, "figures_from_notebook")
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)

# Save PDF + PNG when TRUE. Set FALSE if you only want notebook display.
save_figures <- TRUE
png_dpi <- 300

# ---- Figure sizes in inches: EDIT THESE ----
big_pr_width  <- 13
big_pr_height <- 13

big_roc_width  <- 13
big_roc_height <- 13

aupr_box_width  <- 4.3
aupr_box_height <- 4.3

aupr_paired_width  <- 5.3
aupr_paired_height <- 5.3

auc_box_width  <- 5.3
auc_box_height <- 5.3

auc_paired_width  <- 5.5
auc_paired_height <- 5.3

single_auc_width  <- 5.5
single_auc_height <- 5.5

# Same ordering as the main analysis.
celltypes <- c("Ast", "Exc", "Inh", "Mic_Immune", "OPC", "Oli")
regions   <- c("AG", "EC", "HC", "MTC", "PFC", "TH")

expr_color <- "#E76F51"
act_color  <- "#159D9C"

format_p <- function(p) {
  if (length(p) == 0 || is.na(p) || !is.finite(p)) return("NA")
  if (p < 0.001) return(formatC(p, format = "e", digits = 2))
  formatC(p, format = "f", digits = 3)
}

set_display_size <- function(width, height) {
  options(repr.plot.width = width, repr.plot.height = height, repr.plot.res = 120)
}

save_both <- function(plot_obj, stem, width, height) {
  if (!save_figures) return(invisible(NULL))

  ggsave(
    file.path(figure_dir, paste0(stem, ".pdf")),
    plot_obj,
    width = width,
    height = height,
    units = "in",
    limitsize = FALSE
  )

  ggsave(
    file.path(figure_dir, paste0(stem, ".png")),
    plot_obj,
    width = width,
    height = height,
    units = "in",
    dpi = png_dpi,
    limitsize = FALSE
  )
}

cat("Reading plot data from:\n", output_dir, "\n")
cat("Notebook figures will be saved to:\n", figure_dir, "\n")


## Load plot-ready data

These files are written by the updated main R script, so all plots below can be regenerated without refitting any model.


In [ ]:
summary_df <- read.delim(
  file.path(output_dir, "ALL_celltype_region_AUPR_AUC_summary.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

pr_df <- read.delim(
  file.path(output_dir, "ALL_celltype_region_PR_curve_plotdata.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

roc_df <- read.delim(
  file.path(output_dir, "ALL_celltype_region_ROC_curve_plotdata.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

aupr_long <- read.delim(
  file.path(output_dir, "ALL_celltype_region_AUPR_summary_plotdata.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

auc_long <- read.delim(
  file.path(output_dir, "ALL_celltype_region_AUC_summary_plotdata.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

pair_levels <- unlist(lapply(regions, function(r) paste(celltypes, r, sep = "_")))
pair_levels <- pair_levels[pair_levels %in% summary_df$pair]

summary_df$pair <- factor(summary_df$pair, levels = pair_levels)
pr_df$pair      <- factor(pr_df$pair, levels = pair_levels)
roc_df$pair     <- factor(roc_df$pair, levels = pair_levels)
aupr_long$pair  <- factor(aupr_long$pair, levels = pair_levels)
auc_long$pair   <- factor(auc_long$pair, levels = pair_levels)

aupr_long$Data <- factor(aupr_long$Data, levels = c("Expression", "Activity"))
auc_long$Data  <- factor(auc_long$Data, levels = c("Expression", "Activity"))

pr_df  <- pr_df  %>% arrange(pair, Data, Recall, Precision)
roc_df <- roc_df %>% arrange(pair, Data, FPR, TPR)

wilcox_aupr_p <- wilcox.test(
  summary_df$Activity_AUPR,
  summary_df$Expression_AUPR,
  paired = TRUE,
  alternative = "greater",
  exact = FALSE
)$p.value

wilcox_auc_p <- wilcox.test(
  summary_df$Activity_AUC,
  summary_df$Expression_AUC,
  paired = TRUE,
  alternative = "greater",
  exact = FALSE
)$p.value

cat("Loaded", nrow(summary_df), "celltype-region pairs.\n")
cat("AUPR paired Wilcoxon p (Activity > Expression):", format_p(wilcox_aupr_p), "\n")
cat("AUC paired Wilcoxon p  (Activity > Expression):", format_p(wilcox_auc_p), "\n")


## 1. Big PR / AUPR figure


In [ ]:
annotation_aupr_df <- summary_df %>%
  mutate(
    label = paste0(
      "Expr=", sprintf("%.2f", Expression_AUPR), "\n",
      "Act=", sprintf("%.2f", Activity_AUPR)
    ),
    x = 0.97,
    y = 0.05
  )

baseline_df <- summary_df %>%
  transmute(pair = pair, baseline = lateAD_prevalence)

p_pr_all <- ggplot(
  pr_df,
  aes(x = Recall, y = Precision, color = Data, linetype = Data)
) +
  geom_hline(
    data = baseline_df,
    aes(yintercept = baseline),
    inherit.aes = FALSE,
    linetype = "dashed",
    linewidth = 0.3
  ) +
  geom_step(linewidth = 0.75) +
  geom_text(
    data = annotation_aupr_df,
    aes(x = x, y = y, label = label),
    inherit.aes = FALSE,
    hjust = 1,
    vjust = 0,
    size = 3.2
  ) +
  facet_wrap(~ pair, ncol = 6) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_equal(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE) +
  labs(
    title = "TF expression vs TF activity: nonAD vs lateAD classification",
    x = "Recall",
    y = "Precision",
    color = NULL,
    linetype = NULL,
  ) +
  theme_bw(base_size = 12) +
  theme(
    legend.position = "top",
    strip.text = element_text(face = "bold", size = 12),
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

set_display_size(big_pr_width, big_pr_height)
print(p_pr_all)
save_both(p_pr_all, "ALL_celltype_region_PR_bigplot_notebook", big_pr_width, big_pr_height)


## 2. Big ROC / AUC figure


In [ ]:
annotation_auc_df <- summary_df %>%
  mutate(
    label = paste0(
      "Expr=", sprintf("%.2f", Expression_AUC), "\n",
      "Act=", sprintf("%.2f", Activity_AUC)
    ),
    x = 0.97,
    y = 0.05
  )

p_roc_all <- ggplot(
  roc_df,
  aes(x = FPR, y = TPR, color = Data, linetype = Data)
) +
  geom_abline(
    intercept = 0,
    slope = 1,
    linetype = "dashed",
    linewidth = 0.3
  ) +
  geom_step(linewidth = 0.75) +
  geom_text(
    data = annotation_auc_df,
    aes(x = x, y = y, label = label),
    inherit.aes = FALSE,
    hjust = 1,
    vjust = 0,
    size = 3.2
  ) +
  facet_wrap(~ pair, ncol = 6) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_equal(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE) +
  labs(
    title = "TF expression vs TF activity: nonAD vs lateAD classification",
    x = "False positive rate (1 - specificity)",
    y = "True positive rate (sensitivity)",
    color = NULL,
    linetype = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(
    legend.position = "top",
    strip.text = element_text(face = "bold", size = 8),
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

set_display_size(big_roc_width, big_roc_height)
print(p_roc_all)
save_both(p_roc_all, "ALL_celltype_region_ROC_bigplot_notebook", big_roc_width, big_roc_height)


## 3. Overall AUPR boxplot


In [ ]:
p_aupr_box <- ggplot(
  aupr_long,
  aes(x = Data, y = AUPR, color = Data)
) +
  geom_boxplot(
    width = 0.45,
    outlier.shape = NA,
    linewidth = 0.7,
    alpha = 0
  ) +
  geom_jitter(
    width = 0.08,
    height = 0,
    size = 2,
    alpha = 0.7
  ) +
  annotate(
    "text",
    x = 1.5,
    y = 1.02,
    label = paste0("Paired Wilcoxon p = ", format_p(wilcox_aupr_p)),
    size = 4
  ) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_cartesian(ylim = c(0, 1.08), clip = "off") +
  labs(
    title = "AUPR across all celltype-region datasets",
    subtitle = paste0("n = ", nrow(summary_df), " matched celltype-region pairs"),
    x = NULL,
    y = "AUPR"
  ) +
  theme_classic(base_size = 13) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold"),
    plot.margin = margin(10, 10, 10, 10)
  )

set_display_size(aupr_box_width, aupr_box_height)
print(p_aupr_box)
save_both(p_aupr_box, "ALL_celltype_region_AUPR_boxplot_Wilcoxon_notebook", aupr_box_width, aupr_box_height)


## 4. Overall paired AUPR plot


In [ ]:
p_aupr_paired <- ggplot(
  aupr_long,
  aes(x = Data, y = AUPR, group = pair)
) +
  geom_line(alpha = 0.30, linewidth = 0.5) +
  geom_point(aes(color = Data), size = 2.5, alpha = 0.8) +
  geom_boxplot(
    aes(group = Data, color = Data),
    width = 0.45,
    outlier.shape = NA,
    fill = NA,
    linewidth = 0.7
  ) +
  annotate(
    "text",
    x = 1.5,
    y = 1.08,
    label = paste0("Paired Wilcoxon p = ", format_p(wilcox_aupr_p)),
    size = 5.5
  ) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_cartesian(ylim = c(0, 1.10), clip = "off") +
  labs(
    title = "AUPR comparison",
    subtitle = "Cross cell type-regions",
    x = NULL,
    y = "AUPR"
  ) +
  theme_classic(base_size = 20) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold"),
    plot.margin = margin(10, 10, 10, 10)
  )

set_display_size(5.3, aupr_paired_height)
print(p_aupr_paired)
save_both(p_aupr_paired, "ALL_celltype_region_AUPR_paired_Wilcoxon_notebook", aupr_paired_width, aupr_paired_height)


In [ ]:
library(egg)
p_aupr_paired <- ggplot(
  aupr_long,
  aes(x = Data, y = AUPR, group = pair)
) +
  geom_line(alpha = 0.30, linewidth = 0.5) +
  geom_point(aes(color = Data), size = 2.5, alpha = 0.8) +
  geom_boxplot(
    aes(group = Data, color = Data),
    width = 0.35,
    outlier.shape = NA,
    fill = NA,
    linewidth = 0.7
  ) +
  annotate(
    "text",
    x = 1.5,
    y = 1.08,
    label = paste0("Paired Wilcoxon p = ", format_p(wilcox_aupr_p)),
    size = 5.5
  ) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_cartesian(ylim = c(0, 1.10), clip = "off") +
  labs(
    #title = "AUPR",
    #subtitle = "Cross cell type-regions",
    x = NULL,
    y = "AUPR"
  ) +
  theme_article(base_size = 20) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold"),
    plot.margin = margin(10, 10, 10, 10)
  )

set_display_size(5.3, aupr_paired_height)


aupr_paired_width  <- 4.3
aupr_paired_height <- 4.3

print(p_aupr_paired)
save_both(p_aupr_paired, "ALL_celltype_region_AUPR_paired_Wilcoxon_notebook", aupr_paired_width, aupr_paired_height)


## 5. Overall AUC boxplot


In [ ]:
p_auc_box <- ggplot(
  auc_long,
  aes(x = Data, y = AUC, color = Data)
) +
  geom_boxplot(
    width = 0.35,
    outlier.shape = NA,
    linewidth = 0.7,
    alpha = 0
  ) +
  geom_jitter(
    width = 0.08,
    height = 0,
    size = 2,
    alpha = 0.7
  ) +
  annotate(
    "text",
    x = 1.5,
    y = 1.02,
    label = paste0("Paired Wilcoxon p = ", format_p(wilcox_auc_p)),
    size = 4
  ) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_cartesian(ylim = c(0, 1.08), clip = "off") +
  labs(
    title = "AUC across all celltype-region datasets",
    subtitle = paste0("n = ", nrow(summary_df), " matched celltype-region pairs"),
    x = NULL,
    y = "AUC"
  ) +
  theme_acticle(base_size = 13) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold"),
    plot.margin = margin(10, 10, 10, 10)
  )

set_display_size(auc_box_width, auc_box_height)
print(p_auc_box)
save_both(p_auc_box, "ALL_celltype_region_AUC_boxplot_Wilcoxon_notebook", auc_box_width, auc_box_height)


## 6. Overall paired AUC plot


In [ ]:
p_auc_paired <- ggplot(
  auc_long,
  aes(x = Data, y = AUC, group = pair)
) +
  geom_line(alpha = 0.30, linewidth = 0.5) +
  geom_point(aes(color = Data), size = 2.5, alpha = 0.8) +
  geom_boxplot(
    aes(group = Data, color = Data),
    width = 0.35,
    outlier.shape = NA,
    fill = NA,
    linewidth = 0.7
  ) +
  annotate(
    "text",
    x = 1.5,
    y = 1.08,
    label = paste0("Paired Wilcoxon p = ", format_p(wilcox_auc_p)),
    size = 5.5
  ) +
  scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
  coord_cartesian(ylim = c(0, 1.10), clip = "off") +
  labs(
    #title = "AUC comparison",
    #subtitle = "Cross cell type-regions",
    x = NULL,
    y = "AUC"
  ) +
  theme_article(base_size = 20) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold"),
    plot.margin = margin(10, 10, 10, 10)
  )

set_display_size(5.3, auc_paired_height)

auc_paired_width  <- 4.3
auc_paired_height <- 4.3

print(p_auc_paired)
save_both(p_auc_paired, "ALL_celltype_region_AUC_paired_Wilcoxon_notebook", auc_paired_width, auc_paired_height)


## 7. Plot one specific celltype-region ROC/AUC figure

Edit only `target_celltype` and `target_region` below. For example:

```r
target_celltype <- "Ast"
target_region   <- "AG"
```

The plot uses the **same final LOOCV ROC data** as the 36-panel figure.


In [ ]:
prediction_df <- read.delim(
  file.path(
    output_dir,
    "ALL_celltype_region_LOOCV_predictions.tsv"
  ),
  check.names = FALSE
)

In [ ]:
# ============================================================
# CHOOSE ONE CELLTYPE-REGION HERE
# ============================================================

target_celltype <- "OPC"
target_region   <- "AG"

target_pair <- paste(target_celltype, target_region, sep = "_")

single_roc_df <- roc_df %>%
  filter(as.character(pair) == target_pair)

single_summary <- summary_df %>%
  filter(as.character(pair) == target_pair)

single_pred <- prediction_df %>%
  filter(pair == target_pair)

if (
  nrow(single_roc_df) == 0 ||
  nrow(single_summary) != 1 ||
  nrow(single_pred) == 0
) {
  stop(paste("Could not find exactly one result for", target_pair))
}

expr_auc <- single_summary$Expression_AUC[1]
act_auc  <- single_summary$Activity_AUC[1]

# ------------------------------------------------------------
# Paired DeLong test
# ------------------------------------------------------------

y_single <- ifelse(single_pred$Pathology == "lateAD", 1, 0)

roc_expr_single <- pROC::roc(
  response = y_single,
  predictor = single_pred$Expression_prob,
  levels = c(0, 1),
  direction = "<",
  quiet = TRUE
)

roc_act_single <- pROC::roc(
  response = y_single,
  predictor = single_pred$Activity_prob,
  levels = c(0, 1),
  direction = "<",
  quiet = TRUE
)

delong_p <- tryCatch(
  pROC::roc.test(
    roc_expr_single,
    roc_act_single,
    paired = TRUE,
    method = "delong",
    alternative = "two.sided"
  )$p.value,
  error = function(e) NA_real_
)

delong_label <- ifelse(
  is.na(delong_p),
  "DeLong p = NA",
  ifelse(
    delong_p < 0.001,
    paste0(
      "DeLong p = ",
      formatC(delong_p, format = "e", digits = 2)
    ),
    paste0(
      "DeLong p = ",
      sprintf("%.3f", delong_p)
    )
  )
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

p_single_auc <- ggplot(
  single_roc_df,
  aes(x = FPR, y = TPR, color = Data, linetype = Data)
) +
  geom_abline(
    intercept = 0,
    slope = 1,
    linetype = "dashed",
    linewidth = 0.4
  ) +
  geom_step(linewidth = 1.0) +
  annotate(
    "text",
    x = 0.97,
    y = 0.05,
    hjust = 1,
    vjust = 0,
    label = paste0(
      "Expression AUC = ", sprintf("%.3f", expr_auc), "\n",
      "Activity AUC = ", sprintf("%.3f", act_auc), "\n",
      delong_label
    ),
    size = 4
  ) +
  scale_color_manual(
    values = c(
      "Expression" = expr_color,
      "Activity" = act_color
    )
  ) +
  coord_equal(
    xlim = c(0, 1),
    ylim = c(0, 1),
    expand = FALSE
  ) +
  labs(
    #title = "Expression vs Activity ROC",
    subtitle = target_pair,
    x = "1 - specificity",
    y = "sensitivity",
    color = NULL,
    linetype = NULL
  ) +
  theme_bw(base_size = 20) +
  theme(
    legend.position = "top",
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

set_display_size(single_auc_width, single_auc_height)
print(p_single_auc)

save_both(
  p_single_auc,
  paste0(target_pair, "_ROC_AUC_notebook"),
  single_auc_width,
  single_auc_height
)

## Optional: inspect the exact data used for the selected ROC plot


In [ ]:
single_summary %>%
  select(
    celltype, region, pair,
    n_individuals, n_nonAD, n_lateAD,
    Expression_AUC, Activity_AUC,
    Expression_AUPR, Activity_AUPR
  )

head(single_roc_df)
